# Georgia Food Access ETL Project

## Project Overview 
This project develops an Extract, Transform, Load (ETL) pipeline to clean, standardize, and prepare census tract–level food access data for the state of Georgia using the [USDA Food Access Research Atlas](https://www.ers.usda.gov/data-products/food-access-research-atlas/).

The pipeline transforms raw, high-dimensional data into a validated, analysis-ready dataset by standardizing geographic identifiers (CensusTract and FIPS components), performing data quality checks, and engineering household-level metrics such as SNAP participation rates and vehicle access rates.

Special attention is given to identifying and documenting non-representative tracts, including those with group quarters populations or extremely small populations, to preserve transparency while supporting accurate interpretation.

The final dataset is designed to support analysis of geographic and socioeconomic patterns related to food access across Georgia.

In [7]:
import pandas as pd
import numpy as np

#### Extract Data

In [ ]:
#Load national USDA Food Access Research Atlas dataset
main_df = pd.read_excel("FoodAccessResearchAtlasData2019.xlsx")

#Initial shape check
print("Raw dataset shape:", main_df.shape)

Raw dataset shape: (72531, 147)


#### Filter to Georgia

In [9]:
ga_df = main_df[main_df["State"] == "Georgia"].copy()
print("Georgia dataset shape:", ga_df.shape)

Georgia dataset shape: (1957, 147)


#### Data Inspection

In [10]:
ga_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1957 entries, 18220 to 20176
Columns: 147 entries, CensusTract to TractSNAP
dtypes: float64(126), int64(19), str(2)
memory usage: 2.2 MB


In [11]:
ga_df.head()

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractSeniors,TractWhite,TractBlack,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP
18220,13001950100,Georgia,Appling County,0,3190,1270,0,34.0,1.07,0,...,431.0,2976.0,116.0,7.0,0.0,5.0,86.0,104.0,68.0,164.0
18221,13001950200,Georgia,Appling County,1,4530,1631,0,269.0,5.94,1,...,484.0,2194.0,2027.0,18.0,0.0,5.0,286.0,400.0,184.0,354.0
18222,13001950300,Georgia,Appling County,1,5176,1969,0,123.0,2.38,1,...,770.0,3749.0,778.0,85.0,12.0,12.0,540.0,768.0,251.0,411.0
18223,13001950400,Georgia,Appling County,0,1476,606,0,0.0,0.00,1,...,268.0,1138.0,274.0,8.0,0.0,10.0,46.0,38.0,38.0,96.0
18224,13001950500,Georgia,Appling County,0,3864,1493,0,0.0,0.00,0,...,503.0,3328.0,197.0,8.0,11.0,36.0,284.0,394.0,49.0,246.0


In [12]:
#Validate that all 159 Georgia counties are represented
ga_df["County"].nunique() 

159

In [13]:
#Confirm that each CensusTract appears only once
ga_df['CensusTract'].nunique() == len(ga_df)

True

#### Helper Functions
The following helper functions standardize geographic identifiers and create derived fields. Defining transformations as functions makes the cleaning process more reusable, transparent, and easier to validate.

In [14]:
#Converts Census tract identifiers to 11-character strings.
def standardize_census_tract(df, tract_col="CensusTract"):
    df[tract_col] = df[tract_col].astype(str).str.zfill(11)
    return df

#Splits the 11-digit Census tract GEOID into state, county, and tract components.
def create_geographic_ids(df, tract_col="CensusTract"):
    df["STATEFP"] = df[tract_col].str[:2]
    df["COUNTYFP"] = df[tract_col].str[2:5]
    df["TRACTCE"] = df[tract_col].str[5:]
    return df
    
#Calculates a rate while avoiding division by zero.
def safe_rate(numerator, denominator):
    if denominator == 0 or pd.isna(denominator):
        return np.nan
    return numerator / denominator

#Flags records that may be less representative for tract-level household analysis
def assign_analysis_flag(row):
    reasons = []
    if row["GroupQuartersFlag"] == 1:
        reasons.append("Group quarters")
    if row["Pop2010"] <= 100:
        reasons.append("Small population")
    if reasons:
        return ", ".join(reasons)
    return "Included"

In [15]:
ga_df[ga_df["Pop2010"] <= 100]

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractSeniors,TractWhite,TractBlack,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP
18462,13051010605,Georgia,Chatham County,0,11,5,0,0.0,0.0,0,...,0.0,11.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
18487,13051980000,Georgia,Chatham County,1,2,0,1,2.0,100.0,0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


These helper functions are applied in the following sections to standardize Census tract identifiers, create FIPS-based geographic fields, calculate household-level rates, and flag records that may require special handling during analysis.

#### Standardize Geographic Identifiers

In [16]:
#Standardize CensusTract as an 11-character GEOID
ga_df = standardize_census_tract(ga_df)

#Create STATEFP, COUNTYFP, and TRACTCE fields
ga_df = create_geographic_ids(ga_df)

#Validate standardized tract length
ga_df["CensusTract"].str.len().value_counts()

CensusTract
11    1957
Name: count, dtype: int64

In [17]:
# Confirm all Georgia tract GEOIDs begin with Georgia state FIPS code "13"
ga_df["STATEFP"].value_counts()

STATEFP
13    1957
Name: count, dtype: int64

All CensusTract values were standardized to 11 digits. This is important because geographic identifiers should be stored as strings, not numeric values, to preserve leading zeros and allow accurate joins.

#### County Name Quality Checks
County names are checked for leading/trailing spaces and unexpected double spaces. These checks help confirm that county names are consistent before summary validation.

In [18]:
ga_df[ga_df["County"] != ga_df["County"].str.strip()]

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP,STATEFP,COUNTYFP,TRACTCE


In [19]:
ga_df[ga_df["County"].str.contains("  ", na=False)]

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP,STATEFP,COUNTYFP,TRACTCE


In [20]:
#Visually check correct spelling of GA counties
sorted(ga_df["County"].unique())

['Appling County',
 'Atkinson County',
 'Bacon County',
 'Baker County',
 'Baldwin County',
 'Banks County',
 'Barrow County',
 'Bartow County',
 'Ben Hill County',
 'Berrien County',
 'Bibb County',
 'Bleckley County',
 'Brantley County',
 'Brooks County',
 'Bryan County',
 'Bulloch County',
 'Burke County',
 'Butts County',
 'Calhoun County',
 'Camden County',
 'Candler County',
 'Carroll County',
 'Catoosa County',
 'Charlton County',
 'Chatham County',
 'Chattahoochee County',
 'Chattooga County',
 'Cherokee County',
 'Clarke County',
 'Clay County',
 'Clayton County',
 'Clinch County',
 'Cobb County',
 'Coffee County',
 'Colquitt County',
 'Columbia County',
 'Cook County',
 'Coweta County',
 'Crawford County',
 'Crisp County',
 'Dade County',
 'Dawson County',
 'DeKalb County',
 'Decatur County',
 'Dodge County',
 'Dooly County',
 'Dougherty County',
 'Douglas County',
 'Early County',
 'Echols County',
 'Effingham County',
 'Elbert County',
 'Emanuel County',
 'Evans County',
 '

No leading/trailing whitespace or double-space issues were found in the County field.

#### Data Quality Review: Poverty Rate and Group Quarters
This section investigates extreme poverty values and evaluates whether group quarters populations may distort tract-level analysis. Group quarters include institutional or shared living settings such as dormitories, prisons, and military housing, which may not reflect typical household food access patterns.

In [21]:
ga_df["PovertyRate"].describe()

count    1957.000000
mean       18.331743
std        12.503461
min         0.000000
25%         8.900000
50%        16.200000
75%        25.100000
max       100.000000
Name: PovertyRate, dtype: float64

In [22]:
#Identify tracts with 100% poverty rate
ga_df[ga_df["PovertyRate"] == 100][["CensusTract", "Pop2010", "GroupQuartersFlag", "County"]]

,CensusTract,Pop2010,GroupQuartersFlag,County
18909,13089023115,3228,1,DeKalb County


In [23]:
# Review high-poverty tracts
ga_df[ga_df["PovertyRate"] > 80][[
"CensusTract", "PovertyRate", "GroupQuartersFlag", "Pop2010", "County"]] 

,CensusTract,PovertyRate,GroupQuartersFlag,Pop2010,County
18417,13051000100,83.5,0,1811,Chatham County
18525,13059000100,84.9,0,1596,Clarke County
18527,13059000402,91.2,1,7090,Clarke County
18909,13089023115,100.0,1,3228,DeKalb County


Several census tracts with unusually high poverty rates were examined using map-based context to better understand potential drivers of extreme values.

Includes:

- **13089023115 (DeKalb County)** – Contains a correctional facility, resulting in a high proportion of institutional population and limited reported income.
- **13059000100 and 13059000402 (Clarke County)** – Located near the University of Georgia, with student-heavy populations that report low income, which can inflate poverty statistics.
- **13051000100 (Chatham County)** – A rural tract with geographic isolation and limited economic opportunities.

These findings indicate that extreme poverty values may be driven by non-representative population structures (e.g., institutional or student populations) or geographic constraints, rather than typical household economic conditions.

Because it is not feasible to systematically verify all tracts, these observations are documented rather than used to exclude records from the dataset. These tracts are retained in the dataset to preserve completeness, but users should interpret extreme values with caution.

In [24]:
# Review tracts with 0% poverty rate
ga_df[ga_df["PovertyRate"] == 0][
    ["CensusTract", "PovertyRate", "Pop2010", "GroupQuartersFlag", "County"]
]

,CensusTract,PovertyRate,Pop2010,GroupQuartersFlag,County
18462,13051010605,0.0,11,0,Chatham County
18487,13051980000,0.0,2,1,Chatham County
18492,13053020206,0.0,204,0,Chattahoochee County
19124,13117130604,0.0,4575,0,Forsyth County
19194,13121006801,0.0,2418,1,Fulton County


Review of 0% Poverty Rate Tracts

Some Georgia census tracts reported a PovertyRate of 0%. These records were reviewed because extremely low poverty values can sometimes reflect unusual tract characteristics rather than typical household conditions.

- **13051010605 — Chatham County:** This tract has a very small population (`Pop2010 = 11`), so percentage-based measures may be unstable. It is flagged as a small-population tract.
- **13053020206 — Chattahoochee County:** This tract appears to include a military-base population. Because military populations may not reflect typical civilian household income patterns, this tract is retained but documented for interpretive caution.
- **13117130604 — Forsyth County:** This tract has a larger population (`Pop2010 = 4,575`) and appears consistent with a highly affluent area. It is retained as a valid 0% poverty observation.

These tracts are not automatically removed from the cleaned dataset. Instead, they are documented so users understand that 0% poverty values may represent different underlying conditions depending on tract context.

In [25]:
#Compare average poverty rate by GroupQuartersFlag
ga_df.groupby("GroupQuartersFlag")["PovertyRate"].mean()

GroupQuartersFlag
0    18.258293
1    32.632420
Name: PovertyRate, dtype: float64

In [26]:
#Count group quarters vs non-group quarters tracts
ga_df["GroupQuartersFlag"].value_counts()

GroupQuartersFlag
0    1947
1      10
Name: count, dtype: int64

#### Analysis Flagging and Exclusion Logic
Rather than removing records without explanation, this project creates an analysis flag to document why certain tracts may be excluded from the main analysis. This preserves transparency and allows the full cleaned dataset to remain available

In [27]:
# Create transparent analysis flag before filtering
ga_df["analysis_flag"] = ga_df.apply(assign_analysis_flag, axis=1)

# Review flagged records
ga_df["analysis_flag"].value_counts()

analysis_flag
Included                            1946
Group quarters                         9
Small population                       1
Group quarters, Small population       1
Name: count, dtype: int64

In [28]:
# Create analysis subset excluding special population tracts
ga_analysis = ga_df[ga_df["analysis_flag"] == "Included"].copy()

#### Feature Engineering
Raw counts are converted into rates to allow comparison across census tracts of different sizes. Occupied housing units are used as the denominator for household-level measures such as SNAP participation and vehicle access.

In [29]:
# Create household-level rates for comparison across tracts
ga_df["SNAP_rate"] = ga_df.apply(
    lambda row: safe_rate(row["TractSNAP"], row["OHU2010"]),
    axis=1
)

ga_df["NoVehicle_rate"] = ga_df.apply(
    lambda row: safe_rate(row["TractHUNV"], row["OHU2010"]),
    axis=1
)

In [30]:
# Validate new rate fields
ga_df[["SNAP_rate", "NoVehicle_rate"]].describe()

,SNAP_rate,NoVehicle_rate
count,1956.000000,1956.000000
mean,0.154327,0.078726
std,0.112378,0.084154
min,0.000000,0.000000
25%,0.067486,0.024061
50%,0.136229,0.051389
75%,0.218132,0.103535
max,0.791667,0.633333


In [31]:
# Check for impossible rate values
ga_df[(ga_df["SNAP_rate"] < 0) | (ga_df["SNAP_rate"] > 1)]

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractOMultir,TractHispanic,TractHUNV,TractSNAP,STATEFP,COUNTYFP,TRACTCE,analysis_flag,SNAP_rate,NoVehicle_rate


In [32]:
ga_df[(ga_df["NoVehicle_rate"] < 0) | (ga_df["NoVehicle_rate"] > 1)]

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractOMultir,TractHispanic,TractHUNV,TractSNAP,STATEFP,COUNTYFP,TRACTCE,analysis_flag,SNAP_rate,NoVehicle_rate


#### Create Final Cleaned Dataset
This step selects the final columns needed for analysis and downstream geospatial integration. The final dataset includes geographic identifiers, food access indicators, socioeconomic measures, engineered rates, and analysis flags.

In [33]:
###Tracts containing group quarters populations were excluded from the main analysis, 
### as these populations (e.g., students and incarcerated individuals) do not participate 
### in food access decisions in the same way as households. Including them would artificially 
### inflate poverty rates without reflecting actual food access challenges.
ga_no_gq = ga_df[ga_df["GroupQuartersFlag"] == 0] 

In [34]:
final_cols = [
    "CensusTract",
    "STATEFP",
    "COUNTYFP",
    "TRACTCE",
    "County",
    "Urban",
    "Pop2010",
    "OHU2010",
    "PovertyRate",
    "MedianFamilyIncome",
    "LowIncomeTracts",
    "LILATracts_1And10",
    "LILATracts_Vehicle",
    "TractSNAP",
    "TractHUNV",
    "SNAP_rate",
    "NoVehicle_rate",
    "GroupQuartersFlag",
    "analysis_flag"
]

ga_cleaned = ga_df[final_cols].copy()

In [35]:
ga_cleaned.head()

,CensusTract,STATEFP,COUNTYFP,TRACTCE,County,Urban,Pop2010,OHU2010,PovertyRate,MedianFamilyIncome,LowIncomeTracts,LILATracts_1And10,LILATracts_Vehicle,TractSNAP,TractHUNV,SNAP_rate,NoVehicle_rate,GroupQuartersFlag,analysis_flag
18220,13001950100,13,001,950100,Appling County,0,3190,1270,17.4,68250.0,0,0,0,164.0,68.0,0.129134,0.053543,0,Included
18221,13001950200,13,001,950200,Appling County,1,4530,1631,19.4,45669.0,1,1,1,354.0,184.0,0.217045,0.112814,0,Included
18222,13001950300,13,001,950300,Appling County,1,5176,1969,29.8,41306.0,1,1,1,411.0,251.0,0.208735,0.127476,0,Included
18223,13001950400,13,001,950400,Appling County,0,1476,606,17.0,50000.0,1,1,0,96.0,38.0,0.158416,0.062706,0,Included
18224,13001950500,13,001,950500,Appling County,0,3864,1493,25.2,47258.0,1,0,0,246.0,49.0,0.164769,0.032820,0,Included


#### Final Validation Checks
The final dataset is checked for row counts, missing values, tract uniqueness, and geographic identifier consistency before export.

In [36]:
# Final row and column count
ga_cleaned.shape

(1957, 19)

In [37]:
# Check missing values in final fields
ga_cleaned.isna().sum()

CensusTract            0
STATEFP                0
COUNTYFP               0
TRACTCE                0
County                 0
Urban                  0
Pop2010                0
OHU2010                0
PovertyRate            0
MedianFamilyIncome    12
LowIncomeTracts        0
LILATracts_1And10      0
LILATracts_Vehicle     0
TractSNAP              0
TractHUNV              0
SNAP_rate              1
NoVehicle_rate         1
GroupQuartersFlag      0
analysis_flag          0
dtype: int64

In [38]:
# Confirm CensusTract remains unique
ga_cleaned["CensusTract"].nunique() == len(ga_cleaned)

True

In [39]:
# Confirm all tract IDs are 11 characters
ga_cleaned["CensusTract"].str.len().value_counts()

CensusTract
11    1957
Name: count, dtype: int64

In [40]:
# Confirm all records are Georgia tracts
ga_cleaned["STATEFP"].value_counts()

STATEFP
13    1957
Name: count, dtype: int64

In [41]:
# Confirm county coverage remains complete
ga_cleaned["County"].nunique()

159

In [42]:
# remove small popualtions
ga_filtered_pop = ga_no_gq[ga_no_gq["Pop2010"] > 100] 
# removing tract due to military base skewing poverty rate 
tracts_to_remove = ["13053020206"]
ga_filtered_pop = ga_filtered_pop[~ga_filtered_pop["CensusTract"].isin(tracts_to_remove)]

# Specific tracts identified as non-representative (e.g., extremely small populations or institutional 
# areas) were removed in addition to general population-based filtering

In [43]:
ga_filtered_pop["PovertyRate"].describe()

count    1945.000000
mean       18.277067
std        12.244829
min         0.000000
25%         8.900000
50%        16.200000
75%        25.003011
max        84.900000
Name: PovertyRate, dtype: float64

In [44]:
ga_filtered_pop[ga_filtered_pop["PovertyRate"] == 0]

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractOMultir,TractHispanic,TractHUNV,TractSNAP,STATEFP,COUNTYFP,TRACTCE,analysis_flag,SNAP_rate,NoVehicle_rate
19124,13117130604,Georgia,Forsyth County,1,4575,1432,0,0.0,0.0,0,...,145.0,230.0,0.0,14.0,13,117,130604,Included,0.009777,0.0


In [45]:
ga_filtered_pop[ga_filtered_pop["PovertyRate"] != 0]["PovertyRate"].describe()

# After removing tracts with missing income data and very small populations, 
# the distribution of poverty rates appeared consistent and realistic, with 
# no extreme or implausible minimum values.

count    1944.000000
mean       18.286469
std        12.240956
min         0.700000
25%         8.900000
50%        16.211702
75%        25.027258
max        84.900000
Name: PovertyRate, dtype: float64

In [46]:
ga_filtered_pop["MedianFamilyIncome"].describe() # checking distributiion of median income

count      1940.000000
mean      68402.875258
std       35406.035144
min        9779.000000
25%       44955.500000
50%       58906.000000
75%       81454.500000
max      250001.000000
Name: MedianFamilyIncome, dtype: float64

In [47]:
ga_filtered_pop["TractHUNV"].describe()

count    1945.000000
mean      125.798458
std       124.641443
min         0.000000
25%        39.000000
50%        87.000000
75%       175.000000
max      1121.000000
Name: TractHUNV, dtype: float64

In [48]:
ga_filtered_pop["TractSNAP"].describe()

count    1945.000000
mean      260.549100
std       201.211996
min         0.000000
25%       110.000000
50%       220.000000
75%       367.000000
max      1300.000000
Name: TractSNAP, dtype: float64

In [49]:
# Raw counts of SNAP participation and vehicle access were converted into rates using 
# occupied housing units to allow for meaningful comparison across tracts of different sizes
ga_no_gq["SNAP_rate"] = ga_no_gq["TractSNAP"] / ga_no_gq["OHU2010"]
ga_no_gq["NoVehicle_rate"] = ga_no_gq["TractHUNV"] / ga_no_gq["OHU2010"]

In [50]:
ga_filtered_pop["LILATracts_Vehicle"].value_counts()

LILATracts_Vehicle
0    1444
1     501
Name: count, dtype: int64

In [51]:
ga_filtered_pop["LILATracts_1And10"].value_counts()

LILATracts_1And10
0    1507
1     438
Name: count, dtype: int64

In [52]:
ga_filtered_pop["Urban"].value_counts()

Urban
1    1309
0     636
Name: count, dtype: int64

In [53]:
ga_df_filtered = ga_filtered_pop[[
    "CensusTract",
    "County",
    "Urban",
    "Pop2010",
    "PovertyRate",
    "MedianFamilyIncome",
    "LowIncomeTracts",
    "LILATracts_1And10",
    "LILATracts_Vehicle",
    "SNAP_rate",      
    "NoVehicle_rate", 
    "GroupQuartersFlag"
]].copy()

In [54]:
ga_df_filtered.isna().sum()

CensusTract           0
County                0
Urban                 0
Pop2010               0
PovertyRate           0
MedianFamilyIncome    5
LowIncomeTracts       0
LILATracts_1And10     0
LILATracts_Vehicle    0
SNAP_rate             0
NoVehicle_rate        0
GroupQuartersFlag     0
dtype: int64

In [55]:
ga_df_filtered[ga_df_filtered["MedianFamilyIncome"].isna()]

,CensusTract,County,Urban,Pop2010,PovertyRate,MedianFamilyIncome,LowIncomeTracts,LILATracts_1And10,LILATracts_Vehicle,SNAP_rate,NoVehicle_rate,GroupQuartersFlag
18237,13009970500,Baldwin County,0,7114,50.8,NaN,1,0,0,0.158537,0.062852,0
18455,13051010101,Chatham County,1,917,51.5,NaN,1,0,0,0.306878,0.198413,0
18525,13059000100,Clarke County,1,1596,84.9,NaN,1,1,0,0.035374,0.102041,0
19157,13121002100,Fulton County,1,2451,50.5,NaN,1,0,1,0.185730,0.161948,0
19168,13121003600,Fulton County,1,1207,36.9,NaN,1,0,1,0.241774,0.407725,0


#### Median Family Income Missing Values

A small number of census tracts contain missing values for MedianFamilyIncome. These tracts were reviewed and appear to be associated with:

- Small population sizes
- Institutional or military populations
- Student-heavy areas
- Possible data suppression in Census reporting

Because these missing values likely reflect underlying data limitations rather than errors, they are retained as missing rather than imputed. This preserves data integrity and avoids introducing bias into socioeconomic analysis.

## Final Summary

This ETL pipeline transforms raw USDA Food Access data into a clean, validated, and analysis-ready dataset for Georgia census tracts.

Key contributions include:
- Standardization of geographic identifiers (CensusTract, FIPS components)
- Validation of data integrity through uniqueness, completeness, and consistency checks
- Identification and documentation of non-representative tracts (e.g., group quarters, small populations)
- Creation of household-level metrics such as SNAP_rate and NoVehicle_rate

The final dataset preserves data transparency while supporting meaningful analysis of food access disparities across Georgia.